# 04 — Error analysis

**Goal:** look at *which* test samples the chosen model (logistic regression) gets wrong, not just the aggregate accuracy — aggregate metrics can hide a consistent, explainable failure pattern.

**Input:** the same train/test split as the previous two notebooks.

**Conclusion:** _(written after running the cells below)_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ucu-ai-course/project-template/blob/main/notebooks/04-error-analysis.ipynb)

In [1]:
import sys

# Safe to run locally too: this block only does anything inside Google Colab, where
# the repo isn't already cloned and the package isn't already installed.
if "google.colab" in sys.modules:
    get_ipython().system("git clone https://github.com/ucu-ai-course/project-template.git")
    get_ipython().run_line_magic("cd", "project-template")
    get_ipython().system("pip install -q -r requirements.txt")
    get_ipython().system("pip install -q -e .")


In [2]:
from pathlib import Path
import os

# Notebooks are launched from notebooks/ (`jupyter lab notebooks/`), but every path in this
# project (config/, data/, models/, results/) is written relative to the repo root.
# Running this cell once makes every relative path below "just work", the same way it
# does for the CLI and scripts/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT


PosixPath('/Users/oleksandr/Documents/GitHub/project-template')

In [3]:
import pandas as pd

from wine_origin.config import load_config
from wine_origin.data import (
    load_raw_csv,
    split_features_target,
    train_test_split_stratified,
)
from wine_origin.models import build_active_model
from wine_origin.utils import set_seed

config = load_config("config/default.yaml")
set_seed(config["seed"])

df = load_raw_csv(config["paths"]["raw_data"])
train_df, test_df = train_test_split_stratified(
    df, test_size=config["split"]["test_size"], seed=config["seed"]
)
X_train, y_train = split_features_target(train_df)
X_test, y_test = split_features_target(test_df)

model = build_active_model(config)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
proba = model.predict_proba(X_test)

errors_mask = y_pred != y_test.to_numpy()
int(errors_mask.sum()), len(y_test)

(1, 36)

In [4]:
errors = X_test.loc[errors_mask].copy()
errors["true"] = y_test.to_numpy()[errors_mask]
errors["predicted"] = y_pred[errors_mask]
errors["confidence"] = proba[errors_mask].max(axis=1)
errors[["true", "predicted", "confidence"] + list(X_test.columns[:4])]

,true,predicted,confidence,alcohol,malic_acid,ash,alcalinity_of_ash
1,2,1,0.540423,12.51,1.24,2.25,17.5


In [5]:
# How confident was the model on correct vs. incorrect predictions?
correct_confidence = proba[~errors_mask].max(axis=1)
wrong_confidence = proba[errors_mask].max(axis=1)

pd.DataFrame(
    {
        "correct predictions": pd.Series(correct_confidence).describe(),
        "wrong predictions": pd.Series(wrong_confidence).describe(),
    }
)

,correct predictions,wrong predictions
count,35.000000,1.000000
mean,0.954146,0.540423
std,0.097996,NaN
min,0.600787,0.540423
25%,0.959370,0.540423
50%,0.996635,0.540423
75%,0.998809,0.540423
max,0.999994,0.540423


In [6]:
import matplotlib.pyplot as plt

from wine_origin.evaluate import compute_metrics, plot_confusion_matrix

metrics = compute_metrics(y_test, y_pred)
plot_confusion_matrix(
    metrics["confusion_matrix"],
    ["class_0", "class_1", "class_2"],
    "results/figures/confusion_matrix_error_analysis.png",
)
plt.imread("results/figures/confusion_matrix_error_analysis.png").shape

(600, 750, 4)

## Conclusion

The model misclassifies only a small handful of the 36 test samples (see the count above), and — as the confidence comparison shows — even when it's wrong, its predicted probability for the winning class tends to be lower than on its correct predictions. That's a good sign: the model isn't confidently wrong, it's uncertain on genuinely borderline samples, which lines up with `01-data-exploration.ipynb`'s observation that a few features overlap between classes rather than separating them perfectly. This is also visible in the confusion matrix: errors are concentrated between the two closer classes rather than spread evenly across all pairs.